In [1]:
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.7 MB/s eta 0:00:00


In [2]:
import geemap
import ee


In [3]:
ee.Authenticate()

In [4]:
ee.Initialize(project='plenary-matrix-441317-c4')


In [5]:
#for testing
Map = geemap.Map(center=[56.14735, -3.90733], zoom=9)
Map

Map(center=[56.14735, -3.90733], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

In [6]:
sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR");

USDA_SoilData = ee.ImageCollection("NASA_USDA/HSL/SMAP10KM_soil_moisture");
# 30m Resolution for elevation
NASA_srtm = ee.Image("USGS/SRTMGL1_003");

#Modis
modis_lai_data = ee.ImageCollection('MODIS/061/MCD15A3H');
modis_vegetation_data = ee.ImageCollection("MODIS/061/MYD13Q1")

# Load Landsat 8 and 9 image collections
landsat8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
landsat9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")

# Load the sugarcane and other feature collections
sugarcane_tx = ee.FeatureCollection("projects/plenary-matrix-441317-c4/assets/tx")

In [7]:
sugarcane_tx.size().getInfo()

174

# Calculating Indices from Landset 8 and 9

In [8]:
# Function to cloud mask image
def cloud_mask_landsat(image):
    qa = image.select('QA_PIXEL')
    cloud = 1 << 3
    cirrus = 1 << 9
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask)

In [9]:
# Function to calculate spectral indices
def calculate_indices_landsat(img):

    ndvi = img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    gndvi = img.normalizedDifference(['SR_B5', 'SR_B3']).rename('GNDVI')
    evi = img.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': img.select('SR_B5'),
            'RED': img.select('SR_B4'),
            'BLUE': img.select('SR_B2')
        }
    ).rename('EVI')
    nbr = img.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')
    ndmi = img.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI')
    ndwi = img.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    ndbi = img.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')
    ndbai = img.normalizedDifference(['SR_B6', 'SR_B7']).rename('NDBaI')
    mndwi = img.normalizedDifference(['SR_B3', 'SR_B6']).rename('MNDWI')
    return img.addBands([ndvi,gndvi,evi,nbr, ndmi, ndwi, ndbi, ndbai, mndwi])

In [10]:
landsat8=landsat8.map(cloud_mask_landsat)
landsat8=landsat8.map(calculate_indices_landsat)

landsat9=landsat9.map(cloud_mask_landsat)
landsat9=landsat9.map(calculate_indices_landsat)

# Calculating indices from Sentinel 2 Setallite

In [11]:
def sentinel2_bands_calculations(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }
    ).rename('EVI')
    red = image.select('B4')  # Red band (Sentinel-2 band 4)
    nir = image.select('B8')  # NIR band (Sentinel-2 band 8)
    savi = nir.subtract(red).divide(nir.add(red).add(0.5)).multiply(1.5).rename('SAVI')
    return image.addBands([ndvi, gndvi, evi, savi])

def mask_clouds(image):
    # Using SCL band for Sentinel-2 Level-2A (SR) instead of QA60
    scl = image.select('SCL')
    # 1: Saturated, 3: Shadows, 8-10: Clouds, 11: Snow
    mask = scl.neq(1).And(scl.neq(3)).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask)

def calculate_indices(image):
    # Calculate VCI (Vegetation Condition Index)
    VCI = image.expression('(NDVI - NDVI_min) / (NDVI_max - NDVI_min) * 100',
                           {'NDVI': image.select('NDVI'),
                            'NDVI_min': 0,
                            'NDVI_max': 0.8}).rename('VCI')

    # Calculate Transformed Vegetation Index (TVI)
    TVI = image.expression('0.5 * (NIR - RED) / (NIR + RED + 0.5)',
                           {'NIR': image.select('B8'),
                            'RED': image.select('B4')}).rename('TVI')

    # Calculate Brightness Index (BI)
    BI = image.expression('sqrt((RED**2) + (NIR**2))',
                          {'RED': image.select('B4'),
                           'NIR': image.select('B8')}).rename('BI')

    # Calculate Second Brightness Index (BI2)
    BI2 = image.expression('(RED + NIR) / 2',
                           {'RED': image.select('B4'),
                            'NIR': image.select('B8')}).rename('BI2')

    # Calculate Color Index (CI)
    CI = image.expression('(NIR / RED) - 1',
                          {'NIR': image.select('B8'),
                           'RED': image.select('B4')}).rename('CI')

    # Calculate Clay Index (CI1)
    CI1 = image.expression('(RED / NIR) - 1',
                           {'RED': image.select('B4'),
                            'NIR': image.select('B8')}).rename('CI1')

    # Calculate SATVI (Soil Adjusted Total Vegetation Index)
    SATVI = image.expression('(NIR - RED - 0.5) / (NIR + RED + 0.5)',
                             {'NIR': image.select('B8'),
                              'RED': image.select('B4')}).rename('SATVI')

    # Calculate HVSI (Hue, Value, and Intensity)
    HVSI = image.expression('sqrt((NIR - RED)**2 + (NIR - GREEN)*(RED - GREEN))',
                            {'NIR': image.select('B8'),
                             'RED': image.select('B4'),
                             'GREEN': image.select('B3')}).rename('HVSI')

    # Calculate SOCI (Soil Organic Carbon Index)
    SOCI = image.expression('(NIR / RED) * (1 + RED - GREEN)',
                            {'NIR': image.select('B8'),
                             'RED': image.select('B4'),
                             'GREEN': image.select('B3')}).rename('SOCI')

    # Calculate ASI (Agricultural Stress Index)
    ASI = image.expression('RED - (GREEN + BLUE) / 2',
                           {'RED': image.select('B4'),
                            'GREEN': image.select('B3'),
                            'BLUE': image.select('B2')}).rename('ASI')

    # Calculate BSI (Bare Soil Index)
    BSI = image.expression('1 - (NIR + 0.05) / (RED + 0.05)',
                           {'NIR': image.select('B8'),
                            'RED': image.select('B4')}).rename('BSI')

    # Calculate MSAVI (Modified Soil Adjusted Vegetation Index)
    MSAVI = image.expression('(2 * NIR + 1 - sqrt((2 * NIR + 1)**2 - 8 * (NIR - RED))) / 2',
                             {'NIR': image.select('B8'),
                              'RED': image.select('B4')}).rename('MSAVI')

    return image.addBands([VCI, TVI, BI, BI2, CI, CI1, SATVI, HVSI, SOCI, ASI, BSI, MSAVI])

# Apply functions to collection
sentinel2 = sentinel2.map(mask_clouds)
sentinel2 = sentinel2.map(sentinel2_bands_calculations)
sentinel2 = sentinel2.map(calculate_indices)

In [12]:
import numpy as np
import pandas as pd

In [13]:
# Splitting 43000 into sets so it can be used
total_count = sugarcane_tx.size().getInfo()

# Calculate the number of features in each split
split_count = int(total_count / 10)

# Create a list to store the splits
sugarcane_sets = []

last_index = 0
# Iterate over 10 partitions
for i in range(1, 11):
    # Calculate the starting and ending index for each split
    start_index = (i - 1) * split_count
    end_index = i * split_count


    # Filter the FeatureCollection to get the current split
    split = sugarcane_tx.toList(split_count, start_index)
    sugarcane_sets.append(split)


In [14]:
def normalize_modis_value(x, min_val=-2000, max_val=10000, new_min=-1, new_max=1):
    normalized_value = ((x - min_val) / (max_val - min_val)) * (new_max - new_min) + new_min
    return normalized_value

In [15]:
collection_years = [


    {
        'start_date': '2018-01-01',
        'end_date': '2018-12-31',
        'year':2018,
    },

    {
        'start_date': '2019-01-01',
        'end_date': '2019-12-31',
        'year':2019,
    },

    {
        'start_date': '2021-01-01',
        'end_date': '2021-12-31',
        'year':2021,
    },

    {
        'start_date': '2022-01-01',
        'end_date': '2022-12-31',
        'year':2022,
    }
]

In [16]:
feature_data = []
index = 1

# Standard list of bands to select to ensure homogeneity
s2_standard_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B11', 'B12', 'NDVI', 'EVI', 'GNDVI','SAVI','VCI', 'TVI', 'BI', 'BI2', 'CI', 'CI1', 'SATVI', 'HVSI', 'SOCI', 'ASI', 'BSI', 'MSAVI']

for year in collection_years:

  start_date = year['start_date']
  end_date = year['end_date']

  # We select only the bands we need immediately to avoid homogeneity errors during the median reduction
  sentinel2_data = sentinel2 \
    .filterDate(start_date, end_date) \
    .select(s2_standard_bands)

  landsat8_data = landsat8 \
    .filterDate(start_date, end_date)

  modis_lai = modis_lai_data \
    .filterDate(start_date, end_date)

  modis_vegetation = modis_vegetation_data \
    .filterDate(start_date, end_date)

  USDA_SoilData_data = USDA_SoilData \
    .filterDate(start_date, end_date)

  landsat8_imagery = landsat8_data.mosaic().select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI'])
  sentinel_2_imagery = sentinel2_data.median()
  modis_lai_imagery = modis_lai.mosaic().select(['Lai'])
  modis_vegetation_imagery = modis_vegetation.mosaic().select('NDVI','EVI')
  USDA_imagery = USDA_SoilData_data.mosaic().select(['ssm'])

  field_id = 1

  for set in sugarcane_sets:
    table = set.getInfo()
    for record in table:
      polygon = ee.Feature(record)
      area = polygon.getInfo()['properties']['area']
      roi = polygon.geometry()

      elevation = NASA_srtm.clip(roi).log().divide(10).clamp(0, 1).toFloat()

      landsat8_stats = landsat8_imagery.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7','NBR','NDVI','GNDVI','EVI','NDMI','NDWI','NDBI','NDBaI','MNDWI']) \
        .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30)

      sentinel_2_stats = sentinel_2_imagery \
        .addBands(elevation) \
        .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30)

      modis_lai_stats = modis_lai_imagery.select(['Lai']) \
        .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30)

      modis_vegetation_stats = modis_vegetation_imagery.select(['NDVI','EVI']) \
        .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30)

      USDA_stats = USDA_imagery.select(['ssm']) \
        .reduceRegion(reducer=ee.Reducer.mean(), geometry=roi, scale=30)

      indices = sentinel_2_stats.getInfo()
      modis_lai_indices = modis_lai_stats.getInfo()
      modis_vegetation_indices = modis_vegetation_stats.getInfo()
      usda_indices = USDA_stats.getInfo()
      landsat8_indices = landsat8_stats.getInfo()

      dictionary = {}
      dictionary['field_id'] = field_id
      dictionary['coordinates']= roi.getInfo()['coordinates'][0]
      dictionary['date'] = year['year']

      # Sentinel-2 Indices mapping
      for band in s2_standard_bands:
          dictionary[f'Annual Sentinel 2 {band}'] = indices.get(band)

      dictionary['Annual Sentinel 2 Elevation'] = indices.get('elevation')

      # Landsat Indices mapping
      l8_bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'NBR', 'NDMI', 'NDWI', 'NDBI', 'NDBaI', 'MNDWI', 'NDVI', 'GNDVI', 'EVI']
      for band in l8_bands:
          dictionary[f'Annual {band}_8'] = landsat8_indices.get(band)

      dictionary['Annual MODIS LAI'] = modis_lai_indices.get('Lai')
      dictionary['Annual MODIS NDVI'] = normalize_modis_value(modis_vegetation_indices.get('NDVI'))
      dictionary['Annual MODIS EVI'] = normalize_modis_value(modis_vegetation_indices.get('EVI'))
      dictionary['Annual MOISTURE'] = usda_indices.get('ssm')
      dictionary['Area'] = area

      feature_data.append(dictionary)
      field_id = field_id + 1

    df = pd.DataFrame(feature_data)
    feature_data = []
    df.to_csv(f'Texas_4Y_Records_{index}.csv', index=False)
    from google.colab import files
    files.download(f'Texas_4Y_Records_{index}.csv')
    index = index + 1
    print("Field ID with Year", field_id, start_date)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Field ID with Year 18 2018-01-01


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Field ID with Year 154 2022-01-01


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Field ID with Year 171 2022-01-01


In [17]:
len(feature_data)

0

In [18]:
df = pd.DataFrame(feature_data)

In [ ]:
feature_weights = {
        'field_id': 0,
        'coorrdinates': 0,
        'date': 0,
        'Annual NDVI': 0.25,
        'Annual EVI': 0.2,
        'Annual GNDVI': 0.15,
        'Annual SAVI': 0.15,
        'Annual B1': 0.05,
        'Annual B2': 0.05,
        'Annual B3': 0.05,
        'Annual B4': 0.05,
        'Annual B5': 0.05,
        'Annual B6': 0.05,
        'Annual B7': 0.05,
        'Annual B8': 0.05,
        'Annual B9': 0.05,
        'Annual B11': 0.05,
        'Annual B12': 0.05,
        'Annual SR8_B1': 0.05,
        'Annual SR8_B2': 0.05,
        'Annual SR8_B3': 0.05,
        'Annual SR8_B4': 0.05,
        'Annual SR8_B5': 0.05,
        'Annual SR8_B6': 0.05,
        'Annual SR8_B7': 0.05,
        'Annual NBR_8': 0.03,
        'Annual NDVI_8':0.25,
        'Annual EVI_8':0.2,
        'Annual GNDVI_8':0.15,
        'Annual NDMI_8': 0.04,
        'Annual NDWI_8': 0.04,
        'Annual NDBI_8': 0.03,
        'Annual NDBaI_8': 0.02,
        'Annual MNDWI_8': 0.03,
        'Annual SR9_B1': 0.05,
        'Annual SR9_B2': 0.05,
        'Annual SR9_B3': 0.05,
        'Annual SR9_B4': 0.05,
        'Annual SR9_B5': 0.05,
        'Annual SR9_B6': 0.05,
        'Annual SR9_B7': 0.05,
        'Annual NBR_9': 0.03,
        'Annual NDVI_9':0.25,
        'Annual EVI_9':0.2,
        'Annual GNDVI_9':0.15,
        'Annual NDMI_9': 0.04,
        'Annual NDWI_9': 0.04,
        'Annual NDBI_9': 0.03,
        'Annual NDBaI_9': 0.02,
        'Annual MNDWI_9': 0.03,
        'Annual LAI': 0.2,
        'Annual MOISTURE': 0.1,
        'Area': 0.3,
        'Elevation': 0.1,
        'Annual VCI': 0.1,
        'Annual TVI': 0.1,
        'Annual BI': 0.05,
        'Annual BI2': 0.05,
        'Annual CI': 0.05,
        'Annual CI1': 0.05,
        'Annual SATVI': 0.1,
        'Annual HVSI': 0.1,
        'Annual SOCI': 0.05,
        'Annual ASI': 0.05,
        'Annual BSI': 0.05,
        'Annual MSAVI': 0.1,
        'Annual MODIS NDVI':0.25,
        'Annual MODIS EVI':0.2
    }